# 01 · Ingesta Bronze – Olist Brazilian E-Commerce
**Fuente:** Kaggle – `olistbr/brazilian-ecommerce` (9 archivos CSV)
**Método:** API de Kaggle (`kagglehub`) con token guardado como secreto de Databricks
**Destino:** Volume `olist.bronze.raw_files` → 9 tablas Delta en `olist.bronze`

Bronze guarda el dato **tal como llega** (todo como texto), con columnas de auditoría. La limpieza y los tipos se resuelven en Silver.

In [0]:
%pip install "kagglehub>=0.4.1" --quiet

In [0]:
dbutils.library.restartPython()

## 1. Parámetros
El catálogo se recibe como parámetro para poder ejecutar este notebook desde el Workflow.

In [0]:
dbutils.widgets.text("catalogo", "olist")
catalogo = dbutils.widgets.get("catalogo")
ruta_volume = f"/Volumes/{catalogo}/bronze/raw_files"
print("Catálogo:", catalogo, "| Volume:", ruta_volume)

## 2. Estructura en Unity Catalog
Crea catálogo, esquema y volume si no existen (`IF NOT EXISTS` permite re-ejecutar sin errores).


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo} COMMENT 'Proyecto final: e-commerce Olist'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze COMMENT 'Capa Bronze: datos crudos tal como llegan de Kaggle'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.bronze.raw_files COMMENT 'CSV originales descargados vía API de Kaggle'")
print("Estructura lista")

## 3. Descarga vía API de Kaggle
El token se lee desde el secret scope `kaggle` (nunca queda visible en el código).
Los CSV se descargan a una carpeta temporal y se copian al volume.

In [0]:
import os, shutil, kagglehub

os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kaggle", "api_token")

ruta_descarga = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Descargado en:", ruta_descarga)

csvs = sorted(f for f in os.listdir(ruta_descarga) if f.endswith(".csv"))
for f in csvs:
    shutil.copy(os.path.join(ruta_descarga, f), os.path.join(ruta_volume, f))
    print("Copiado:", f)
print(f"\nTotal: {len(csvs)} archivos")

In [0]:
display(dbutils.fs.ls(ruta_volume))

## 4. Carga a tablas Delta Bronze
- `inferSchema=False`: todo se guarda como texto, sin alterar el origen.
- `multiLine` + `escape`: necesarios por los comentarios con saltos de línea en `order_reviews`.
- Columnas de auditoría: `_ingestion_ts` (cuándo entró) y `_source_file` (de qué archivo vino).
- Se eliminan caracteres invisibles (BOM) de los encabezados.

In [0]:
from pyspark.sql import functions as F

archivos = {
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

def limpiar_nombre(c):
    return c.replace("\ufeff", "").strip().lower().replace(" ", "_")

def ingestar(tabla, archivo):
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .load(f"{ruta_volume}/{archivo}"))

    df = (df.withColumn("_ingestion_ts", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path")))
    df = df.toDF(*[limpiar_nombre(c) for c in df.columns])

    nombre = f"{catalogo}.bronze.{tabla}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(nombre))
    return nombre, spark.table(nombre).count()

In [0]:
resultados = []
for tabla, archivo in archivos.items():
    nombre, filas = ingestar(tabla, archivo)
    print(f"{nombre}: {filas:,} filas")
    resultados.append((nombre, filas))

resumen = (spark.createDataFrame(resultados, ["tabla", "filas"])
           .withColumn("fecha_ingesta", F.current_timestamp()))
resumen.write.format("delta").mode("append").saveAsTable(f"{catalogo}.bronze.log_ingesta")
display(resumen)

## 5. Verificación
Historial de versiones Delta y registro acumulado de ingestas.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {catalogo}.bronze.orders"))
display(spark.sql(f"SELECT * FROM {catalogo}.bronze.log_ingesta ORDER BY fecha_ingesta DESC"))tabla